# 量子認知モデル by UI/UX最適化
## Quantum Cognition Model for UI/UX Optimization

**研究概要:**  
Wang & Busemeyer (2014) の量子認知モデルをUI/UXのCVR（コンバージョン率）順序効果に応用する。

**このノートブックの内容:**
1. 射影演算子（Projection Operators）の定義
2. 2要素モデル：A→B と B→A の順序効果シミュレーション
3. QQ等式（Quantum Question Equality）の検証
4. 3要素モデルへの拡張
5. 古典モデルとの比較

**参考文献:**  
Wang, Z., & Busemeyer, J. R. (2013). A quantum question order model supported by empirical tests of an a priori and precise prediction. *Topics in Cognitive Science*, 5(4), 689–710.

In [ ]:
# ============================================================
# セル1：ライブラリのインポート
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy.optimize import minimize, curve_fit
from itertools import permutations
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定（Google Colab用）
try:
    import japanize_matplotlib
except ImportError:
    !pip install japanize-matplotlib -q
    import japanize_matplotlib

matplotlib.rcParams['figure.dpi'] = 120
print('✅ ライブラリ読み込み完了')

---
## Part 1：射影演算子の定義

### 理論背景

ユーザーの心理状態を2次元ヒルベルト空間 $\mathcal{H}^2$ 上のベクトル $|\psi\rangle$ で表現する。

$$|\psi\rangle = \cos\frac{\phi}{2}|0\rangle + \sin\frac{\phi}{2}|1\rangle$$

UIブロック $X$（機能説明・価格・実績など）を読む行為を、角度 $\theta_X$ の射影演算子 $P_X$ で表現する：

$$P_X = \begin{pmatrix} \cos^2\theta_X & \cos\theta_X\sin\theta_X \\ \cos\theta_X\sin\theta_X & \sin^2\theta_X \end{pmatrix}$$

UIブロック $X$ を読んだ後に「購買意欲あり」と判断する確率：

$$p(X) = \langle\psi|P_X|\psi\rangle$$

In [ ]:
# ============================================================
# セル2：量子認知モデルのコアクラス
# ============================================================

class QuantumCognitionModel:
    """
    UI/UX向け量子認知モデル
    
    Parameters
    ----------
    psi_angle : float
        初期心理状態ベクトルの角度 φ（ラジアン）
        φ=0 → 完全に「購買意欲なし」状態
        φ=π/2 → 完全に「購買意欲あり」状態
    """
    
    def __init__(self, psi_angle: float = np.pi / 4):
        self.phi = psi_angle
        self.psi = np.array([np.cos(self.phi / 2),
                             np.sin(self.phi / 2)])  # 初期状態ベクトル |ψ⟩

    # ----------------------------------------------------------
    # 1. 射影演算子
    # ----------------------------------------------------------
    @staticmethod
    def projection_matrix(theta: float) -> np.ndarray:
        """
        角度 θ の射影演算子 P を返す（2×2行列）
        P = |e⟩⟨e| where |e⟩ = (cos θ, sin θ)
        """
        c, s = np.cos(theta), np.sin(theta)
        return np.array([[c*c, c*s],
                         [c*s, s*s]])

    # ----------------------------------------------------------
    # 2. 単一UIブロックを読んだ後の確率
    # ----------------------------------------------------------
    def prob_single(self, theta: float) -> float:
        """
        UIブロック X（角度θ）単体を読んだ後に「購買意欲あり」となる確率
        p(X) = ⟨ψ|P_X|ψ⟩
        """
        P = self.projection_matrix(theta)
        return float(self.psi @ P @ self.psi)

    # ----------------------------------------------------------
    # 3. 2ブロックの連続読み（順序効果あり）
    # ----------------------------------------------------------
    def prob_sequential(self, theta_first: float, theta_second: float) -> float:
        """
        UIブロックを first → second の順に読んだ後の確率
        p(first, second) = ⟨ψ|P_first P_second P_first|ψ⟩
        
        ※ 量子力学では射影後に状態が更新されるため順序が重要
        """
        P1 = self.projection_matrix(theta_first)
        P2 = self.projection_matrix(theta_second)
        # |ψ'⟩ = P1|ψ⟩ / ||P1|ψ⟩||  （P1で射影後の状態に規格化）
        psi_after_first = P1 @ self.psi
        norm_sq = float(psi_after_first @ psi_after_first)  # ||P1|ψ⟩||²
        if norm_sq < 1e-12:
            return 0.0
        # P2による射影確率（P1適用後の状態から）
        psi_proj = P2 @ psi_after_first
        return float(psi_proj @ psi_proj) / norm_sq * norm_sq
        # = ⟨ψ|P1 P2 P1|ψ⟩

    # ----------------------------------------------------------
    # 4. 任意のn個のUIブロック順列に対する確率
    # ----------------------------------------------------------
    def prob_sequence(self, thetas: list) -> float:
        """
        UIブロックを thetas の順に読んだ後の「購買意欲あり」確率
        p = ⟨ψ| P_n...P_2 P_1 P_1 P_2...P_n |ψ⟩
        """
        # 射影演算子の積を計算
        P_total = np.eye(2)
        for theta in thetas:
            P_total = self.projection_matrix(theta) @ P_total
        result_vec = P_total @ self.psi
        return float(result_vec @ result_vec)


# 動作確認
model = QuantumCognitionModel(psi_angle=np.pi/4)
theta_A = np.pi / 6   # UIブロックA（機能説明）
theta_B = np.pi / 3   # UIブロックB（価格）

print('=== 基本動作確認 ===')
print(f'P_A（θ=π/6）=\n{model.projection_matrix(theta_A).round(3)}')
print(f'P_B（θ=π/3）=\n{model.projection_matrix(theta_B).round(3)}')
print(f'p(A) = {model.prob_single(theta_A):.4f}')
print(f'p(B) = {model.prob_single(theta_B):.4f}')
print(f'p(A→B) = {model.prob_sequence([theta_A, theta_B]):.4f}')
print(f'p(B→A) = {model.prob_sequence([theta_B, theta_A]):.4f}')
print('\n✅ 順序効果が存在するか:', 
      abs(model.prob_sequence([theta_A, theta_B]) - 
          model.prob_sequence([theta_B, theta_A])) > 1e-6)

---
## Part 2：QQ等式（Quantum Question Equality）の検証

Wang & Busemeyer (2013) が導出したQQ等式：

$$p(A=1, B=1) + p(A=0, B=0) - p(B=1, A=1) - p(B=0, A=0) = 0$$

**この等式が成立 ⟺ 量子モデルで記述可能**  
古典確率モデルでは一般にこの等式は成立しない。

In [ ]:
# ============================================================
# セル3：QQ等式の計算と検証
# ============================================================

def compute_qq_equality(model: QuantumCognitionModel,
                        theta_A: float,
                        theta_B: float) -> dict:
    """
    QQ等式の各項を計算して返す
    
    Returns
    -------
    dict with keys:
        p_AB_11: p(A=Yes, B=Yes) — A→Bの順でどちらもYes
        p_AB_00: p(A=No,  B=No)  — A→Bの順でどちらもNo
        p_BA_11: p(B=Yes, A=Yes) — B→Aの順でどちらもYes
        p_BA_00: p(B=No,  B=No)  — B→Aの順でどちらもNo
        qq_value: QQ等式の左辺の値（理論値は0）
    """
    P_A = model.projection_matrix(theta_A)
    P_B = model.projection_matrix(theta_B)
    # 補射影演算子（Noに対応）
    Q_A = np.eye(2) - P_A
    Q_B = np.eye(2) - P_B
    psi = model.psi

    def born(operator):
        """ボルン則: ⟨ψ|M|ψ⟩"""
        v = operator @ psi
        return float(v @ v)

    # A→B の順
    p_AB_11 = born(P_B @ P_A)          # A=Yes → B=Yes
    p_AB_00 = born(Q_B @ Q_A)          # A=No  → B=No

    # B→A の順
    p_BA_11 = born(P_A @ P_B)          # B=Yes → A=Yes
    p_BA_00 = born(Q_A @ Q_B)          # B=No  → A=No

    qq_value = (p_AB_11 + p_AB_00) - (p_BA_11 + p_BA_00)

    return {
        'p_AB_11': p_AB_11,
        'p_AB_00': p_AB_00,
        'p_BA_11': p_BA_11,
        'p_BA_00': p_BA_00,
        'qq_value': qq_value
    }


# --- QQ等式の確認 ---
result = compute_qq_equality(model, theta_A, theta_B)
print('=== QQ等式の検証 ===')
for k, v in result.items():
    print(f'  {k:12s} = {v:.6f}')
print(f'\n理論値との誤差: {abs(result["qq_value"]):.2e}')
print('✅ QQ等式成立（|値| < 1e-10）:', abs(result['qq_value']) < 1e-10)


# --- 全角度範囲でQQ等式を検証（数値的証明） ---
print('\n=== 全角度でのQQ等式の成立確認 ===')
max_violation = 0
for phi in np.linspace(0, np.pi, 20):
    for tA in np.linspace(0, np.pi, 20):
        for tB in np.linspace(0, np.pi, 20):
            m = QuantumCognitionModel(psi_angle=phi)
            r = compute_qq_equality(m, tA, tB)
            max_violation = max(max_violation, abs(r['qq_value']))
print(f'  最大違反量: {max_violation:.2e}（数値誤差のみ）')
print('  → 量子モデルはQQ等式を常に満たす ✅')

---
## Part 3：順序効果の可視化

θ_A と θ_B の組み合わせに対して、どの程度の順序効果が生じるかを可視化する。

In [ ]:
# ============================================================
# セル4：順序効果のヒートマップ
# ============================================================

phi = np.pi / 4  # 初期状態（中立）
model = QuantumCognitionModel(psi_angle=phi)

thetas = np.linspace(0, np.pi/2, 50)
order_effect = np.zeros((len(thetas), len(thetas)))

for i, tA in enumerate(thetas):
    for j, tB in enumerate(thetas):
        p_AB = model.prob_sequence([tA, tB])
        p_BA = model.prob_sequence([tB, tA])
        order_effect[i, j] = p_AB - p_BA  # 正 → A先が有利、負 → B先が有利

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ヒートマップ
im = axes[0].imshow(order_effect, origin='lower', cmap='RdBu',
                    vmin=-0.3, vmax=0.3,
                    extent=[0, 90, 0, 90])
plt.colorbar(im, ax=axes[0], label='p(A→B) - p(B→A)')
axes[0].set_xlabel('θ_B（度）')
axes[0].set_ylabel('θ_A（度）')
axes[0].set_title('順序効果 p(A→B) - p(B→A)\n（赤：A先が有利、青：B先が有利）')
axes[0].plot([0,90],[0,90], 'k--', alpha=0.4, lw=1, label='θ_A=θ_B（効果なし）')
axes[0].legend(fontsize=8)

# 特定のθ_Aに対するθ_B依存性
for tA_deg, color in [(15,'royalblue'), (30,'tomato'), (45,'green'), (60,'purple')]:
    tA = np.radians(tA_deg)
    effects = []
    for tB in thetas:
        p_AB = model.prob_sequence([tA, tB])
        p_BA = model.prob_sequence([tB, tA])
        effects.append(p_AB - p_BA)
    axes[1].plot(np.degrees(thetas), effects, color=color,
                 label=f'θ_A = {tA_deg}°', lw=2)

axes[1].axhline(0, color='gray', lw=1, linestyle='--')
axes[1].set_xlabel('θ_B（度）')
axes[1].set_ylabel('順序効果の大きさ')
axes[1].set_title('θ_B に対する順序効果（φ=45°）')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('order_effect_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 図を order_effect_heatmap.png に保存しました')

---
## Part 4：3要素モデル（A・B・Cの全6順列）

UIブロック A（機能）・B（価格）・C（実績）の全6通りの提示順序に対する CVR予測

In [ ]:
# ============================================================
# セル5：3要素モデルと全6順列の予測
# ============================================================

# UIブロックの角度パラメータ（論文では実験データからフィッティングして推定）
THETA = {
    'A': np.radians(20),   # 機能説明ブロック
    'B': np.radians(50),   # 価格・プランブロック
    'C': np.radians(35),   # 実績・レビューブロック
}

model_3 = QuantumCognitionModel(psi_angle=np.pi / 4)

# 全6順列の確率を計算
print('=== 3要素全順列の予測CVR ===')
print(f'{"順序":<8} {"予測CVR":>10} {"順位":>6}')
print('-' * 28)

results_3 = []
for perm in permutations(['A', 'B', 'C']):
    thetas = [THETA[x] for x in perm]
    prob = model_3.prob_sequence(thetas)
    results_3.append((perm, prob))

results_3.sort(key=lambda x: x[1], reverse=True)
for rank, (perm, prob) in enumerate(results_3, 1):
    print(f'{"-".join(perm):<8} {prob:>10.4f} {rank:>6}')

# 棒グラフ
labels = ['-'.join(p) for p, _ in results_3]
values = [v for _, v in results_3]
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(labels)))[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylabel('予測CVR（購買意欲あり確率）')
ax.set_title('UIブロック提示順序と予測CVR（量子認知モデル）')
ax.set_ylim(0, max(values) * 1.2)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cvr_3block_prediction.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 図を cvr_3block_prediction.png に保存しました')

---
## Part 5：古典モデルとの比較 & パラメータフィッティング

実験データが得られた後、θパラメータを推定するフィッティング関数と、古典モデルとの比較を行う。

In [ ]:
# ============================================================
# セル6：パラメータフィッティング（実験データ用）
# ============================================================

def fit_quantum_model(observed_cvrs: dict, initial_params=None):
    """
    実験データ（各順列のCVR）からθパラメータを推定する
    
    Parameters
    ----------
    observed_cvrs : dict
        例: {('A','B','C'): 0.35, ('A','C','B'): 0.28, ...}
    initial_params : list or None
        初期値 [phi, theta_A, theta_B, theta_C]
    
    Returns
    -------
    dict: 推定パラメータ, フィット誤差, モデル予測値
    """
    if initial_params is None:
        initial_params = [np.pi/4, np.pi/6, np.pi/3, np.pi/4]

    def residuals(params):
        phi, tA, tB, tC = params
        m = QuantumCognitionModel(psi_angle=phi)
        theta_map = {'A': tA, 'B': tB, 'C': tC}
        total_sq_err = 0
        for seq, obs in observed_cvrs.items():
            thetas = [theta_map[x] for x in seq]
            pred = m.prob_sequence(thetas)
            total_sq_err += (pred - obs) ** 2
        return total_sq_err

    # 最適化（パラメータの範囲: 0 ≤ 各角度 ≤ π）
    bounds = [(0, np.pi)] * 4
    result = minimize(residuals, initial_params, method='L-BFGS-B', bounds=bounds)

    phi_opt, tA_opt, tB_opt, tC_opt = result.x
    m_opt = QuantumCognitionModel(psi_angle=phi_opt)
    theta_map = {'A': tA_opt, 'B': tB_opt, 'C': tC_opt}

    predictions = {}
    for seq in observed_cvrs:
        thetas = [theta_map[x] for x in seq]
        predictions[seq] = m_opt.prob_sequence(thetas)

    return {
        'phi':     np.degrees(phi_opt),
        'theta_A': np.degrees(tA_opt),
        'theta_B': np.degrees(tB_opt),
        'theta_C': np.degrees(tC_opt),
        'rmse':    np.sqrt(result.fun / len(observed_cvrs)),
        'predictions': predictions,
        'success': result.success
    }


# --- ダミーデータでフィッティングのテスト ---
# ※ 実際の実験後はここを実測データに置き換える
dummy_data = {
    ('A','B','C'): 0.38,
    ('A','C','B'): 0.31,
    ('B','A','C'): 0.29,
    ('B','C','A'): 0.24,
    ('C','A','B'): 0.35,
    ('C','B','A'): 0.27,
}

fit_result = fit_quantum_model(dummy_data)

print('=== パラメータフィッティング結果 ===')
print(f'  φ（初期状態）  = {fit_result["phi"]:.1f}°')
print(f'  θ_A（機能）    = {fit_result["theta_A"]:.1f}°')
print(f'  θ_B（価格）    = {fit_result["theta_B"]:.1f}°')
print(f'  θ_C（実績）    = {fit_result["theta_C"]:.1f}°')
print(f'  RMSE           = {fit_result["rmse"]:.4f}')
print(f'  収束           = {fit_result["success"]}')

print('\n=== 観測値 vs 予測値 ===')
print(f'{"順序":<10} {"観測CVR":>10} {"予測CVR":>10} {"誤差":>8}')
print('-' * 42)
for seq, obs in dummy_data.items():
    pred = fit_result['predictions'][seq]
    print(f'{"-".join(seq):<10} {obs:>10.3f} {pred:>10.3f} {pred-obs:>+8.3f}')

In [ ]:
# ============================================================
# セル7：古典モデル（独立確率）との比較
# ============================================================

class ClassicalIndependentModel:
    """
    古典的な独立確率モデル（ベースライン比較用）
    各ブロックの効果は独立で、順序に関係なく同じ確率を仮定する
    p(A→B→C) = p_A * p_B * p_C（順序効果なし）
    """
    def __init__(self, p_A: float, p_B: float, p_C: float):
        self.probs = {'A': p_A, 'B': p_B, 'C': p_C}

    def prob_sequence(self, seq):
        """順序に関係なく、各ブロックの確率の積を返す"""
        result = 1.0
        for x in seq:
            result *= self.probs[x]
        return result


def compare_models(observed_cvrs: dict):
    """量子モデルと古典モデルをRMSEで比較"""

    # 量子モデル
    quantum_fit = fit_quantum_model(observed_cvrs)
    quantum_rmse = quantum_fit['rmse']

    # 古典モデル：各ブロックの平均CVRを初期値として最適化
    all_labels = ['A', 'B', 'C']
    def classical_residuals(params):
        pA, pB, pC = params
        cm = ClassicalIndependentModel(pA, pB, pC)
        return sum((cm.prob_sequence(seq) - obs)**2
                   for seq, obs in observed_cvrs.items())

    classical_result = minimize(classical_residuals, [0.7, 0.6, 0.7],
                                bounds=[(0,1),(0,1),(0,1)], method='L-BFGS-B')
    classical_rmse = np.sqrt(classical_result.fun / len(observed_cvrs))

    print('=== モデル比較 ===')
    print(f'  量子モデル  RMSE: {quantum_rmse:.5f}')
    print(f'  古典モデル  RMSE: {classical_rmse:.5f}')
    improvement = (classical_rmse - quantum_rmse) / classical_rmse * 100
    print(f'  RMSEの改善: {improvement:.1f}%')
    print()
    if quantum_rmse < classical_rmse:
        print('→ 量子モデルの方が実験データをより良く説明する ✅')
    else:
        print('→ 古典モデルの方が良い（量子効果が弱い可能性）')

    return quantum_rmse, classical_rmse


q_rmse, c_rmse = compare_models(dummy_data)

# 棒グラフで比較
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['古典モデル\n（独立確率）', '量子認知モデル'],
       [c_rmse, q_rmse],
       color=['#e74c3c', '#2ecc71'], alpha=0.85, width=0.5)
ax.set_ylabel('RMSE（小さいほど良い）')
ax.set_title('モデル適合度の比較')
ax.set_ylim(0, max(c_rmse, q_rmse) * 1.4)
for i, v in enumerate([c_rmse, q_rmse]):
    ax.text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 図を model_comparison.png に保存しました')

---
## まとめ・次のステップ

| ノートブック | 内容 |
|---|---|
| `01_model.ipynb`（本ファイル） | 数理モデル定義・QQ等式・シミュレーション |
| `02_simulation.ipynb` | 実験前の感度分析・サンプルサイズ計算 |
| `03_analysis.ipynb` | 実測データの読み込み・フィッティング・検定 |

### 実験後にやること
1. `dummy_data` を実測データに差し替えて `fit_quantum_model()` を再実行
2. `compare_models()` で古典モデルとの優劣を判定
3. QQ等式を実測データで検証（`compute_qq_equality()` を使用）

### 注意事項
- θパラメータは実験データから推定するものであり、事前に決定する値ではない
- 各順列あたり最低60サンプルを確保すること（検出力分析より）
- 中間測定（各ブロック後の好感度スコア）を取得すると射影の証拠が強化される